# Chapter 2
I will be using The Verdict by Edith Wharton.

**-Perform version checks**

In [2]:
from importlib.metadata import version


print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

torch version: 2.5.1
tiktoken version: 0.14.0


**-Load the story's text file**

In [3]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
    
print("Total # of characters:", len(raw_text))
print(raw_text[:99])

Total # of characters: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


**-Now split based on whitespace and punctuation/symbols, deleting all empty tokens**

In [4]:
import re
preprocessed = re.split(r'([,.:;?_!"()\'-]|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(len(preprocessed)) #4863
print(preprocessed[:30])

4863
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '-', '-', 'though', 'a', 'good', 'fellow', 'enough', '-', '-', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that']


**Note that in order to convert tokens into token IDs, we build a vocabulary to map each unique word to a unique integer value.**

**-We create a vocabulary where each unique token is added in alphabetical order.**

In [5]:
uniqueWords = sorted(set(preprocessed))
print(len(uniqueWords)) #1139 unique words

1139


In [6]:
vocab = {token:integer for integer, token in enumerate(uniqueWords)} #token-integer pairs made
for i, item in enumerate(vocab.items()):
    print(item) #we show the first 20 pairs as a check
    if i >= 20: break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('-', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)


**-We also want the ability to convert IDs back to text. Lets make a class to both encode text and decode IDs.**

In [7]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\'-]|--|\s)', text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'-])', r'\1', text)
        return text

**We test the abilities of the tokenizer with a small string.**

In [8]:
text = "I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would have been Rome or Florence.)"


tokenizer = SimpleTokenizerV1(vocab)
ids = tokenizer.encode(text)
print(ids)

print(tokenizer.decode(ids))


[53, 44, 149, 1012, 57, 38, 825, 115, 258, 492, 6, 6, 1011, 115, 506, 440, 397, 6, 6, 916, 588, 1087, 713, 512, 970, 1025, 666, 1025, 537, 996, 5, 571, 997, 540, 726, 551, 502, 5, 535, 518, 375, 551, 752, 5, 664, 115, 848, 1111, 5, 157, 402, 549, 571, 115, 1076, 731, 997, 84, 7, 3, 99, 53, 825, 1012, 588, 1129, 532, 206, 85, 738, 34, 7, 4]
I HAD always thought Jack Gisburn rather a cheap genius-- though a good fellow enough-- so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera.( Though I rather thought it would have been Rome or Florence.)


**-We need to add special tokens which indicate unknown words, and end of text. We'll make another tokenizer SimpleTokenizerV2:**

In [9]:
#we add two special tokens endoftext and unk  into our tokens and add it to our vocab
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token:integer for integer, token in enumerate(all_tokens)}
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1136)
('your', 1137)
('yourself', 1138)
('<|endoftext|>', 1139)
('<|unk|>', 1140)


In [10]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int 
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [11]:
#trying out the new tokenizer which utilizes unknown word and endoftext tokens
text1 = "one two three four five six"
text2 = "a b c d e f"
text = " <|endoftext|> ".join((text1, text2))
print(text)

#print results
tokenizer = SimpleTokenizerV2(vocab)
print(tokenizer.encode(text))
print(tokenizer.decode(tokenizer.encode(text)))

#Also consider other special tokens such as beginning of sequence BOS, end of sequence EOS and padding PAD tokens

one two three four five six <|endoftext|> a b c d e f
[733, 1140, 1013, 476, 450, 1140, 1139, 115, 1140, 1140, 315, 1140, 1140]
one <|unk|> three four five <|unk|> <|endoftext|> a <|unk|> <|unk|> d <|unk|> <|unk|>


**-2.5: Byte Pair Encoding**

We'll use tiktoken to avoid the complexity of byte pair encoding

In [12]:
from importlib.metadata import version
import tiktoken
print(version("tiktoken"))

0.14.0


In [13]:
tokenizer = tiktoken.get_encoding("gpt2")


**-Example usage of the BPE tokenizer**

Note that the BPE already has a vocab of 50,257 words, and for unknown words it breaks it down into substrings (down to the character) and encodes the substrings.

In [14]:
tokenizer = tiktoken.get_encoding("gpt2")
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

strings = tokenizer.decode(integers)
print(strings)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]
Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


**-Data sampling and pretraining**

We implement a data loader which fetches input-target pairs (i.e. current and next word pairs)

In [15]:
with open("the-verdict.txt", "r", encoding = "utf-8") as f:
    raw_text=f.read()
encoded = tokenizer.encode(raw_text)
print(len(encoded)) #5145 tokens

5145


**-For demonstration purposes, remove first 50 tokens to get to a more interesting text passage (what?)**

We'll also create vectors containing the input and target pairs, which are just the words offset by one.

In [16]:
sample = encoded[:50]
contextSize = 4
x = sample[:contextSize]
y = sample[1:contextSize+1]
print(f"x: {x}")
print(f"y:      {y}")

x: [40, 367, 2885, 1464]
y:      [367, 2885, 1464, 1807]


**-And now we create the next-word prediction tasks**

In [17]:
for i in range(1, contextSize+1):
    context = sample[:i]
    desired = sample[i]
    print(context, "---->", desired)

print("\n Now in words...\n")

for i in range(1, contextSize+1):
    context = sample[:i]
    desired = sample[i]
    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

[40] ----> 367
[40, 367] ----> 2885
[40, 367, 2885] ----> 1464
[40, 367, 2885, 1464] ----> 1807

 Now in words...

I ---->  H
I H ----> AD
I HAD ---->  always
I HAD always ---->  thought


**-We need to load our token pairs into tensors for the LLM to actually start learning. We can create a data loader to do so:**

In [18]:
import torch
from torch.utils.data import Dataset, DataLoader
print("PyTorch version:", torch.__version__)


PyTorch version: 2.5.1


In [ ]:
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text  
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length+1"

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]



#dataloader
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

**-We test our dataloader with batch size 1, context size 4**

In [30]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
    
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


**-To see the effect of stride, we fetch another batch from the dataset**

In [28]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


-As seen, stride dictates how many positions the inputs shift

In general, smaller context size yields more noisy data as model sometimes doesn't have enough context to predict the next word accurately

too large of a context size yields diminishing returns in model understanding, not necessary and spikes computational power needed

a good middle ground is preferred as the model can properly predict the next word but also isn't super computationally demanding

Additionally, too small of a stride may cause overfitting due to repeated items being trained on multiple times. It may be best to have stride = context length, as though this causes some context loss it improves computational efficiency and reduces overfitting risk.

In [31]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


**-The final step in preprocessing is to convert token IDs to embedding vectors, which allow neural networks to actually learn via backprop**

As an example for embedding, assume we have a tensor of 4 tokens:

In [32]:
input_ids = torch.tensor([2,3,5,1])
#lets say we have a small vocab of size 6 and want embeddings of size 3
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size,output_dim)
print(embedding_layer.weight)


Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


**-Embedding size is essentialy the # of dimensions each token gets to match contexts. Higher dimensions/sizes means more rich learning**

In [34]:
print(embedding_layer(torch.tensor([3])))
print(embedding_layer(input_ids))


tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)
tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)
